# ⚡ FreeCompute — Universal Dual-GPU Model Server

**Local-first AI pair programming powered by free remote GPU compute.**

- **Default Model:** Huihui Qwen3.8-27B-Abliterated Q4 on Kaggle Dual Tesla T4 GPUs (30 GB VRAM).
- **Universal Architecture:** Swap in Qwen 2.5 Coder 32B, DeepSeek-R1 Distill 32B, Mistral, GLM, or any GGUF from Hugging Face by uncommenting presets in **Cell 2**.
- **Security:** Model runs remotely on Kaggle; all file inspection, edits, approvals, and terminal execution stay 100% strictly local on your PC.

### Kaggle Session Settings:
- **Accelerator:** `GPU T4 x2`
- **Internet:** `ON`
- **Persistence:** `None` or `Variables and Files`

---
### Quick Instructions:
1. Verify **Accelerator: GPU T4 x2** and **Internet: ON** in the right panel.
2. Select your model preset in **Cell 2** (default is Qwen 3.8-27B Abliterated).
3. Click **Run All** (or run cells 1 through 8 in sequence).
4. Copy the `freecompute --remote-url ...` command printed in **Cell 8** and run it in your local terminal!

In [ ]:
# ==============================================================================
# 1. MODEL SELECTION & RUNTIME CONFIGURATION
# Uncomment any preset below to switch models, or specify any Hugging Face GGUF!
# ==============================================================================
import secrets

# ------------------------------------------------------------------------------
# PRESET 1: Qwen 3.8-27B Abliterated (DEFAULT)
# Uncensored / abliterated, strong reasoning and coding, 64K context.
# ------------------------------------------------------------------------------
MODEL_PRESET = {
    "REPO_ID": "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF",
    "FILENAME": "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf",
    "REVISION": "3f101cd22b7999228bbd5d79a33975414eb9758b",
    "MODEL_ALIAS": "qwen3.8-27b-huihui-abliterated-q4",
    "CTX_SIZE": 65536,
}

# ------------------------------------------------------------------------------
# PRESET 2: Qwen 2.5 Coder 32B Instruct
# Premier open coding model for deep architectural refactoring.
# ------------------------------------------------------------------------------
# MODEL_PRESET = {
#     "REPO_ID": "Qwen/Qwen2.5-Coder-32B-Instruct-GGUF",
#     "FILENAME": "qwen2.5-coder-32b-instruct-q4_k_m.gguf",
#     "REVISION": "main",
#     "MODEL_ALIAS": "qwen2.5-coder-32b",
#     "CTX_SIZE": 49152,
# }

# ------------------------------------------------------------------------------
# PRESET 3: DeepSeek-R1 Distill Qwen 32B
# Intense reasoning and mathematical problem solving.
# ------------------------------------------------------------------------------
# MODEL_PRESET = {
#     "REPO_ID": "unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF",
#     "FILENAME": "DeepSeek-R1-Distill-Qwen-32B-Q4_K_M.gguf",
#     "REVISION": "main",
#     "MODEL_ALIAS": "deepseek-r1-distill-32b",
#     "CTX_SIZE": 32768,
# }

# ------------------------------------------------------------------------------
# PRESET 4: Mistral Small 24B Instruct 2501
# Fast, low-latency, great conversational agent.
# ------------------------------------------------------------------------------
# MODEL_PRESET = {
#     "REPO_ID": "bartowski/mistral-small-24b-instruct-2501-GGUF",
#     "FILENAME": "mistral-small-24b-instruct-2501-Q4_K_M.gguf",
#     "REVISION": "main",
#     "MODEL_ALIAS": "mistral-small-24b",
#     "CTX_SIZE": 32768,
# }

CONFIG = {
    **MODEL_PRESET,

    # Dual-GPU layer split settings (50/50 across both Tesla T4s)
    "N_GPU_LAYERS": 999,       # Offload all layers to GPUs
    "SPLIT_MODE": "layer",     # 'layer' splits layers across GPUs (CUDA0 + CUDA1)
    "TENSOR_SPLIT": "1,1",     # 50/50 layer balance across dual T4s
    "BATCH_SIZE": 512,
    "UBATCH_SIZE": 128,
    "CACHE_TYPE_K": "f16",
    "CACHE_TYPE_V": "f16",

    # Ports & Security Gateway
    "SUPERVISOR_PORT": 8081,
    "LLAMA_PORT": 8080,
    "API_KEY": secrets.token_hex(16),  # Session Bearer token

    # Network Transport Bridge: 'cloudflare' (default, zero setup) or 'tailscale'
    "TRANSPORT": "cloudflare",
    "TAILSCALE_AUTHKEY": "",   # Optional: paste your tskey-auth-... if using Tailscale
}

print(f"Configured model : {CONFIG['MODEL_ALIAS']}")
print(f"Hugging Face repo: {CONFIG['REPO_ID']}")
print(f"Target context   : {CONFIG['CTX_SIZE']} tokens on dual-GPU {CONFIG['SPLIT_MODE']} split ({CONFIG['TENSOR_SPLIT']})")

In [ ]:
# ==============================================================================
# 2. HARDWARE & SCRATCH PREFLIGHT
# ==============================================================================
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path('/kaggle/working')
SCRATCH = Path('/kaggle/tmp/freecompute')
MODELS_DIR = SCRATCH / 'models'
BUILD_DIR = SCRATCH / 'llama.cpp' / 'build'

SCRATCH.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(SCRATCH / 'hf_home')

# Verify GPUs
smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free', '--format=csv,noheader'],
    text=True, capture_output=True, check=True
).stdout.strip()
print('Detected GPUs:\n' + smi)

cuda_bin = Path('/usr/local/cuda/bin')
if cuda_bin.is_dir():
    os.environ['PATH'] = str(cuda_bin) + os.pathsep + os.environ.get('PATH', '')

for exe in ('git', 'cmake', 'nvcc'):
    if not shutil.which(exe):
        raise RuntimeError(f'Required executable {exe!r} is missing.')

free_scratch_gb = shutil.disk_usage(SCRATCH).free / (1024**3)
print(f'Free scratch disk: {free_scratch_gb:.1f} GiB')
if free_scratch_gb < 20:
    raise RuntimeError('Less than 20 GiB scratch space available.')
print('PREFLIGHT CHECKS PASSED.')

In [ ]:
# ==============================================================================
# 3. REUSABLE DATASET DETECTION (FAST START vs COLD BUILD)
# ==============================================================================
cached_server = None
cached_model = None

input_dir = Path('/kaggle/input')
if input_dir.is_dir():
    for p in input_dir.glob('**/llama-server'):
        if p.is_file() and os.access(p, os.X_OK):
            cached_server = p
            break
    for p in input_dir.glob(f'**/{CONFIG["FILENAME"]}'):
        if p.is_file():
            cached_model = p
            break

SERVER_BIN = None
MODEL_PATH = None

if cached_server:
    print(f'FAST START: Found precompiled llama-server at {cached_server}')
    local_bin = SCRATCH / 'bin' / 'llama-server'
    local_bin.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached_server, local_bin)
    local_bin.chmod(0o755)
    SERVER_BIN = local_bin
else:
    print('COLD START: No precompiled llama-server in /kaggle/input; will build from source.')

if cached_model:
    print(f'FAST START: Found cached model checkpoint at {cached_model}')
    MODEL_PATH = cached_model
else:
    print(f'COLD START: Model {CONFIG["FILENAME"]} not in /kaggle/input; will download from Hugging Face.')

In [ ]:
# ==============================================================================
# 4. ENGINE BUILD (PINNED COMMIT 2b129cc WITH -DGGML_CUDA_NO_VMM=ON)
# ==============================================================================
if SERVER_BIN is None or not SERVER_BIN.is_file():
    SOURCE_DIR = SCRATCH / 'llama.cpp'
    if not SOURCE_DIR.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/ggml-org/llama.cpp.git', str(SOURCE_DIR)
        ], check=True)
    
    commit = subprocess.run(
        ['git', '-C', str(SOURCE_DIR), 'rev-parse', 'HEAD'],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    print('Building llama.cpp commit:', commit)

    # Auto-detect CUDA compute capability (75 for T4, 80 for A100, 89 for L4)
    cuda_arch = "75"
    try:
        smi_cap = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            text=True
        ).splitlines()[0].strip().replace('.', '')
        if smi_cap:
            cuda_arch = smi_cap
    except Exception:
        pass
    print(f'Targeting CUDA architecture: sm_{cuda_arch}')

    BUILD = SOURCE_DIR / 'build'
    subprocess.run([
        'cmake',
        '-S', str(SOURCE_DIR),
        '-B', str(BUILD),
        '-DGGML_CUDA=ON',
        '-DGGML_CUDA_NO_VMM=ON',
        f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}',
        '-DCMAKE_BUILD_TYPE=Release',
        '-DLLAMA_BUILD_TESTS=OFF',
    ], check=True)

    print('Compiling llama-server with --parallel 2...')
    subprocess.run([
        'cmake', '--build', str(BUILD),
        '--target', 'llama-server',
        '--parallel', '2'
    ], check=True)

    SERVER_BIN = BUILD / 'bin' / 'llama-server'
    if not SERVER_BIN.is_file():
        raise RuntimeError('Build failed to produce llama-server binary.')
    print('ENGINE BUILD SUCCEEDED.')

subprocess.run([str(SERVER_BIN), '--version'], check=True)
print('llama-server ready at:', SERVER_BIN)

In [ ]:
# ==============================================================================
# 5. MODEL CHECKPOINT RETRIEVAL
# ==============================================================================
import sys
if MODEL_PATH is None or not Path(MODEL_PATH).is_file():
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'huggingface_hub>=0.34,<1.0', 'hf_xet>=1.0', 'requests>=2.28',
    ], check=True)
    from huggingface_hub import HfApi, hf_hub_download

    print(f'Downloading {CONFIG["FILENAME"]} from {CONFIG["REPO_ID"]}...')
    download_kwargs = {
        'repo_id': CONFIG['REPO_ID'],
        'filename': CONFIG['FILENAME'],
        'local_dir': str(MODELS_DIR),
    }
    if CONFIG.get('REVISION'):
        download_kwargs['revision'] = CONFIG['REVISION']

    downloaded = hf_hub_download(**download_kwargs)
    MODEL_PATH = Path(downloaded)
    size_gb = MODEL_PATH.stat().st_size / 1e9
    print(f'Downloaded {MODEL_PATH.name}: {size_gb:.2f} GB')

print('MODEL READY AT:', MODEL_PATH)

In [ ]:
# ==============================================================================
# 6. START LLAMA-SERVER & AUTHENTICATED SUPERVISOR
# ==============================================================================
import time
import requests

LOG_PATH = WORK / 'llama_server.log'
SUPERVISOR_PY = SCRATCH / 'supervisor.py'

supervisor_script = '#!/usr/bin/env python3\n"""\nkaggle/supervisor.py — Lightweight Authenticated Supervisor & Watchdog for llama-server\nRuns inside the Kaggle GPU container, providing:\n1. Token-based authentication (Bearer token gate on all endpoints).\n2. Health & Telemetry endpoint (/health) with real container uptime and dual-GPU stats.\n3. Streaming OpenAI-compatible reverse proxy (/v1/chat/completions) to llama-server.\n4. Process lifecycle & watchdog to monitor, restart, and safely terminate llama-server.\n"""\n\nimport hmac\nimport re\nimport argparse\nimport http.server\nimport json\nimport os\nimport shutil\nimport socketserver\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\n\nclass SupervisorConfig:\n    def __init__(self, port=8081, llama_port=8080, api_key=None, llama_cmd=None, log_path=""):\n        self.port = port\n        self.llama_port = llama_port\n        self.api_key = os.environ.get("SUPERVISOR_API_KEY") if api_key is None else api_key\n        if not self.api_key or not self.api_key.strip():\n            raise ValueError("SUPERVISOR_API_KEY must be nonempty")\n        self.llama_cmd = llama_cmd or []\n        self.log_path = log_path or "/kaggle/working/llama_server.log"\n        self.start_time = time.time()\n        self.llama_process = None\n        self.watchdog_active = True\n\n\nconfig = None\n\ndef scrub(text):\n    text = str(text)\n    if config and config.api_key:\n        text = text.replace(config.api_key, "[REDACTED_SECRET]")\n    text = re.sub(r"(?i)Bearer\\s+\\S+", "Bearer [REDACTED_TOKEN]", text)\n    text = re.sub(r"https?://[^\\s]+", "[REDACTED_URL]", text)\n    return text\n\n\ndef get_container_uptime_seconds():\n    """Extract real container uptime from /proc/uptime if available."""\n    try:\n        with open("/proc/uptime", "r") as f:\n            return float(f.readline().split()[0])\n    except Exception:\n        return time.time() - config.start_time\n\n\ndef get_gpu_telemetry():\n    """Query nvidia-smi for dual Tesla T4 memory, temperature, and utilization."""\n    if not shutil.which("nvidia-smi"):\n        return []\n    try:\n        cmd = [\n            "nvidia-smi",\n            "--query-gpu=index,name,memory.used,memory.total,temperature.gpu,utilization.gpu",\n            "--format=csv,noheader,nounits",\n        ]\n        out = subprocess.check_output(cmd, text=True, stderr=subprocess.DEVNULL).strip()\n        gpus = []\n        for line in out.splitlines():\n            parts = [p.strip() for p in line.split(",")]\n            if len(parts) >= 6:\n                gpus.append({\n                    "index": int(parts[0]),\n                    "name": parts[1],\n                    "vramUsedMiB": int(parts[2]),\n                    "vramTotalMiB": int(parts[3]),\n                    "tempC": int(parts[4]),\n                    "utilizationPct": int(parts[5]),\n                })\n        return gpus\n    except Exception as exc:\n        return [{"error": scrub(exc)}]\n\n\ndef check_llama_health():\n    """Check if the internal llama-server answers on its loopback port."""\n    url = f"http://127.0.0.1:{config.llama_port}/health"\n    try:\n        req = urllib.request.Request(url, method="GET")\n        with urllib.request.urlopen(req, timeout=3) as resp:\n            return resp.status == 200\n    except Exception:\n        return False\n\n\ndef start_llama_server():\n    """Launch the llama-server subprocess if a command was configured."""\n    if not config.llama_cmd:\n        return None\n    print(f"[SUPERVISOR] Starting llama-server on port {config.llama_port}...")\n    log_dir = os.path.dirname(config.log_path)\n    if log_dir:\n        os.makedirs(log_dir, exist_ok=True)\n    log_file = open(config.log_path, "a", encoding="utf-8")\n    proc = subprocess.Popen(\n        config.llama_cmd,\n        stdout=log_file,\n        stderr=subprocess.STDOUT,\n    )\n    config.llama_process = proc\n    return proc\n\n\nclass SupervisorHandler(http.server.BaseHTTPRequestHandler):\n    protocol_version = "HTTP/1.1"\n\n    def log_message(self, format, *args):\n        sys.stderr.write(scrub(f"[SUPERVISOR {self.address_string()}] {format % args}") + "\\n")\n\n    def check_auth(self):\n        """Validate Authorization: Bearer <API_KEY> header."""\n        if config is None or not config.api_key:\n            self.send_error_response(503, "Supervisor authentication is not configured")\n            return False\n        auth_header = self.headers.get("Authorization", "")\n        if not auth_header.startswith("Bearer "):\n            self.send_error_response(401, "Missing or invalid Authorization header. Expected Bearer token.")\n            return False\n        token = auth_header[7:].strip()\n        if not hmac.compare_digest(token.encode(), config.api_key.encode()):\n            self.send_error_response(403, "Forbidden: Invalid API Key.")\n            return False\n        return True\n\n    def send_error_response(self, status_code, message):\n        body = json.dumps({"error": {"message": scrub(message), "code": status_code}}).encode("utf-8")\n        self.send_response(status_code)\n        self.send_header("Content-Type", "application/json")\n        self.send_header("Content-Length", str(len(body)))\n        self.end_headers()\n        self.wfile.write(body)\n\n    def do_GET(self):\n        if not self.check_auth():\n            return\n        parsed = urllib.parse.urlparse(self.path)\n        path = parsed.path\n\n        if path == "/health":\n            llama_ok = check_llama_health()\n            container_uptime = time.time() - config.start_time\n            gpus = get_gpu_telemetry()\n            payload = {\n                "status": "healthy" if llama_ok else "degraded",\n                "supervisorUptimeSeconds": round(time.time() - config.start_time, 1),\n                "sessionAgeSeconds": round(container_uptime, 1),\n                "sessionAgeSource": "supervisor_start_estimate",\n                "isEstimate": True,\n                "linuxUptimeSeconds": round(get_container_uptime_seconds(), 1),\n                "maxSessionSeconds": 43200,\n                "secondsRemainingIn12hSession": max(0, round(43200 - container_uptime, 1)),\n                "llamaServer": {\n                    "healthy": llama_ok,\n                    "port": config.llama_port,\n                    "pid": config.llama_process.pid if config.llama_process else None,\n                },\n                "gpus": gpus,\n            }\n            body = json.dumps(payload, indent=2).encode("utf-8")\n            self.send_response(200 if llama_ok else 503)\n            self.send_header("Content-Type", "application/json")\n            self.send_header("Content-Length", str(len(body)))\n            self.end_headers()\n            self.wfile.write(body)\n            return\n\n        if not self.check_auth():\n            return\n\n        if path == "/v1/models" and not parsed.query:\n            self.proxy_to_llama("GET")\n        else:\n            self.send_error_response(404, f"Endpoint {path} not found.")\n\n    def do_POST(self):\n        parsed = urllib.parse.urlparse(self.path)\n        path = parsed.path\n\n        if not self.check_auth():\n            return\n\n        if path == "/control/restart" and not parsed.query:\n            if not config.llama_cmd:\n                self.send_error_response(409, "Supervisor does not own a restart command")\n                return\n            if config.llama_process and config.llama_process.poll() is None:\n                config.llama_process.terminate()\n                try:\n                    config.llama_process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    config.llama_process.kill()\n            start_llama_server()\n            body = json.dumps({"status": "restarting", "message": "llama-server restart signal sent."}).encode("utf-8")\n            self.send_response(200)\n            self.send_header("Content-Type", "application/json")\n            self.send_header("Content-Length", str(len(body)))\n            self.end_headers()\n            self.wfile.write(body)\n            return\n\n        if path == "/v1/chat/completions" and not parsed.query:\n            self.proxy_to_llama("POST")\n        else:\n            self.send_error_response(404, f"Endpoint {path} not found.")\n\n    def proxy_to_llama(self, method):\n        target_url = f"http://127.0.0.1:{config.llama_port}{self.path}"\n        try:\n            content_length = int(self.headers.get("Content-Length", 0))\n            if not 0 <= content_length <= 16 * 1024 * 1024 or self.headers.get("Transfer-Encoding"):\n                raise ValueError()\n        except ValueError:\n            self.send_error_response(400, "Invalid request body length")\n            return\n        body = self.rfile.read(content_length) if content_length > 0 else None\n\n        headers = {}\n        for k, v in self.headers.items():\n            if k.lower() not in ("host", "authorization", "content-length"):\n                headers[k] = v\n        if content_length > 0:\n            headers["Content-Length"] = str(content_length)\n\n        req = urllib.request.Request(target_url, data=body, headers=headers, method=method)\n\n        try:\n            with urllib.request.urlopen(req, timeout=900) as resp:\n                self.send_response(resp.status)\n                is_sse = False\n                for k, v in resp.headers.items():\n                    if k.lower() == "transfer-encoding" and v.lower() == "chunked":\n                        continue\n                    if k.lower() == "content-type" and "text/event-stream" in v.lower():\n                        is_sse = True\n                    self.send_header(k, v)\n                self.send_header("Connection", "close")\n                self.close_connection = True\n                self.end_headers()\n\n                while True:\n                    chunk = resp.readline() if is_sse else resp.read(8192)\n                    if not chunk:\n                        break\n                    try:\n                        self.wfile.write(chunk)\n                        self.wfile.flush()\n                    except (BrokenPipeError, ConnectionResetError):\n                        break\n        except urllib.error.HTTPError as err:\n            self.send_error_response(err.code, "Internal inference backend returned an error")\n        except Exception:\n            self.send_error_response(502, "Internal inference backend unavailable")\n\n\nclass ThreadedHTTPServer(socketserver.ThreadingMixIn, http.server.HTTPServer):\n    daemon_threads = True\n    allow_reuse_address = True\n\n\ndef run_supervisor(port=8081, llama_port=8080, api_key=None, llama_cmd=None, log_path=""):\n    global config\n    config = SupervisorConfig(port, llama_port, api_key, llama_cmd, log_path)\n\n    if config.llama_cmd:\n        start_llama_server()\n\n    server_address = ("0.0.0.0", config.port)\n    httpd = ThreadedHTTPServer(server_address, SupervisorHandler)\n    try:\n        httpd.serve_forever()\n    except KeyboardInterrupt:\n        pass\n    finally:\n        httpd.server_close()\n        if config.llama_process and config.llama_process.poll() is None:\n            config.llama_process.terminate()\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Kaggle llama-server Authenticated Supervisor")\n    parser.add_argument("--port", type=int, default=8081, help="Supervisor external port (default: 8081)")\n    parser.add_argument("--llama-port", type=int, default=8080, help="Internal llama-server port (default: 8080)")\n    parser.add_argument("--api-key", type=str, default=None, help="Bearer token for API access")\n    parser.add_argument("--log-path", type=str, default="/kaggle/working/llama_server.log", help="Log output file path")\n    args = parser.parse_args()\n\n    run_supervisor(port=args.port, llama_port=args.llama_port, api_key=args.api_key, log_path=args.log_path)\n'
SUPERVISOR_PY.write_text(supervisor_script, encoding='utf-8')

# Start llama-server
llama_cmd = [
    str(SERVER_BIN),
    '--model', str(MODEL_PATH),
    '--alias', CONFIG['MODEL_ALIAS'],
    '--host', '127.0.0.1',
    '--port', str(CONFIG['LLAMA_PORT']),
    '--n-gpu-layers', str(CONFIG['N_GPU_LAYERS']),
    '--split-mode', CONFIG['SPLIT_MODE'],
    '--tensor-split', CONFIG['TENSOR_SPLIT'],
    '--ctx-size', str(CONFIG['CTX_SIZE']),
    '--parallel', '1',
    '--fit', 'off',
    '--batch-size', str(CONFIG['BATCH_SIZE']),
    '--ubatch-size', str(CONFIG['UBATCH_SIZE']),
    '--cache-type-k', CONFIG['CACHE_TYPE_K'],
    '--cache-type-v', CONFIG['CACHE_TYPE_V'],
    '--flash-attn', 'auto',
    '--jinja',
    '--no-context-shift',
]

llama_env = os.environ.copy()
if SERVER_BIN:
    lib_dir = str(Path(SERVER_BIN).parent)
    llama_env['LD_LIBRARY_PATH'] = f"{lib_dir}:{llama_env.get('LD_LIBRARY_PATH', '')}"

with open(LOG_PATH, 'w', encoding='utf-8') as f:
    llama_proc = subprocess.Popen(llama_cmd, stdout=f, stderr=subprocess.STDOUT, env=llama_env)

print('Booting llama-server...')
time.sleep(5)

# Start supervisor
env = os.environ.copy()
env['SUPERVISOR_API_KEY'] = CONFIG['API_KEY']
env['LLAMA_PORT'] = str(CONFIG['LLAMA_PORT'])
supervisor_proc = subprocess.Popen(
    [sys.executable, str(SUPERVISOR_PY), '--port', str(CONFIG['SUPERVISOR_PORT']), '--llama-port', str(CONFIG['LLAMA_PORT'])],
    env=env
)

print(f'Waiting for llama-server model load on port {CONFIG["LLAMA_PORT"]} (up to 8 mins)...')
healthy = False
for i in range(96):
    try:
        r = requests.get(f'http://127.0.0.1:{CONFIG["SUPERVISOR_PORT"]}/health', headers={'Authorization': 'Bearer ' + CONFIG['API_KEY']}, timeout=2)
        if r.status_code == 200 and r.json().get('llamaServer', {}).get('healthy'):
            healthy = True
            break
    except Exception:
        pass
    if i % 6 == 0:
        print(f'Loading model layers onto dual T4 GPUs... {(i + 1) * 5}s')
    time.sleep(5)

if not healthy:
    print('Failed to start. Server log tail:')
    subprocess.run(['tail', '-n', '50', str(LOG_PATH)], check=False)
    raise RuntimeError('Model failed to become healthy.')

print('*** SUPERVISOR HEALTHY ON PORT', CONFIG['SUPERVISOR_PORT'], '***')
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total', '--format=csv,noheader'], check=True)

In [ ]:
# ==============================================================================
# 7. SECURE TRANSPORT BRIDGE (CLOUDFLARE HTTP/2 OR TAILSCALE)
# ==============================================================================
import os, re, shutil, subprocess, time
from pathlib import Path

SCRATCH = globals().get('SCRATCH', Path('/kaggle/tmp/freecompute'))
WORK = globals().get('WORK', Path('/kaggle/working'))
SUPERVISOR_PORT = CONFIG.get('SUPERVISOR_PORT', 8081)
API_KEY = CONFIG.get('API_KEY', '')

use_tailscale = CONFIG.get('TRANSPORT') == 'tailscale' and CONFIG.get('TAILSCALE_AUTHKEY')

if use_tailscale:
    print('Starting Tailscale userspace node...')
    TS_DIR = SCRATCH / 'tailscale'
    TS_DIR.mkdir(parents=True, exist_ok=True)
    TSD = TS_DIR / 'tailscaled'
    TS_CLI = TS_DIR / 'tailscale'
    if not TSD.is_file():
        subprocess.run([
            'curl', '-fsSL',
            'https://pkgs.tailscale.com/stable/tailscale_1.76.6_amd64.tgz',
            '-o', str(SCRATCH / 'tailscale.tgz')
        ], check=True)
        subprocess.run([
            'tar', '-xzf', str(SCRATCH / 'tailscale.tgz'),
            '--strip-components=1', '-C', str(TS_DIR)
        ], check=True)

    ts_daemon = subprocess.Popen([
        str(TSD),
        '--tun=userspace-networking',
        '--socks5-server=localhost:1055',
        f'--state={SCRATCH}/tailscaled.state',
        f'--socket={SCRATCH}/tailscaled.sock',
    ])
    time.sleep(3)

    subprocess.run([
        str(TS_CLI), f'--socket={SCRATCH}/tailscaled.sock',
        'up', f'--authkey={CONFIG["TAILSCALE_AUTHKEY"]}',
        '--hostname=kaggle-freecompute-brain',
        '--accept-routes'
    ], check=True)
    status = subprocess.check_output([
        str(TS_CLI), f'--socket={SCRATCH}/tailscaled.sock', 'ip', '-4'
    ], text=True).strip()
    print('=' * 70)
    print('  ⚡ FREECOMPUTE REMOTE GPU SUPERVISOR ONLINE (TAILSCALE)')
    print('=' * 70)
    print(f'  Tailscale IP : http://{status}:{SUPERVISOR_PORT}')
    print(f'  API Key      : {API_KEY}')
    print('=' * 70)
    print('\nRun locally in PowerShell:')
    print(f'freecompute --remote-url "http://{status}:{SUPERVISOR_PORT}" --api-key "{API_KEY}"')

else:
    # 1. Kill any existing tunnel to prevent conflicts
    subprocess.run(['pkill', '-9', '-f', 'cloudflared'], check=False)
    time.sleep(1)

    # 2. Download cloudflared if needed
    cf_bin = SCRATCH / 'cloudflared'
    if not cf_bin.is_file():
        print('Downloading cloudflared...')
        subprocess.run([
            'curl', '-fsSL', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-o', str(cf_bin)
        ], check=True)
        cf_bin.chmod(0o755)

    # 3. Start cloudflared with --protocol http2 (enforces TCP port 443, preventing UDP disconnects)
    cf_log = WORK / 'cloudflared.log'
    with open(cf_log, 'w') as f:
        cf_proc = subprocess.Popen([
            str(cf_bin), 'tunnel',
            '--protocol', 'http2',
            '--no-autoupdate',
            '--url', f'http://127.0.0.1:{SUPERVISOR_PORT}'
        ], stdout=f, stderr=subprocess.STDOUT)

    print('Waiting for Cloudflare tunnel URL...')
    tunnel_url = None
    for _ in range(30):
        time.sleep(1)
        if cf_log.is_file():
            text = cf_log.read_text(encoding='utf-8', errors='ignore')
            matches = re.findall(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', text)
            if matches:
                tunnel_url = matches[0]
                break

    if tunnel_url:
        print('\n' + '=' * 70)
        print('  ⚡ FREECOMPUTE REMOTE GPU SUPERVISOR ONLINE (HTTP/2)')
        print('=' * 70)
        print(f'  Public URL : {tunnel_url}')
        print(f'  API Key    : {API_KEY}')
        print('=' * 70)
        print('\nRun locally in PowerShell:')
        print(f'freecompute --remote-url "{tunnel_url}" --api-key "{API_KEY}"')
    else:
        print('Tunnel log tail:')
        subprocess.run(['tail', '-n', '25', str(cf_log)], check=False)
        raise RuntimeError('Cloudflare tunnel did not produce a URL in 30 seconds.')

In [ ]:
# ==============================================================================
# 8. VERIFICATION TEST (LOCAL PROXY & TOOL CALL TEST)
# ==============================================================================
import json

base_url = f"http://127.0.0.1:{CONFIG['SUPERVISOR_PORT']}"
headers = {'Authorization': f"Bearer {CONFIG['API_KEY']}"}

# 1. Health check
health_res = requests.get(base_url + '/health')
print('Supervisor Health:', health_res.json())

# 2. Generation smoke test
gen_payload = {
    'model': CONFIG['MODEL_ALIAS'],
    'messages': [
        {'role': 'system', 'content': 'You are a coding assistant. Return only Python code.'},
        {'role': 'user', 'content': 'Write a function multiply(x, y) that returns their product.'},
    ],
    'max_tokens': 64,
    'temperature': 0,
}
resp = requests.post(base_url + '/v1/chat/completions', json=gen_payload, headers=headers)
print('HTTP Response:', resp.status_code)
print('Model Generation Test:\n', resp.json()['choices'][0]['message']['content'])
print('VERIFICATION PASSED.')

In [ ]:
# ==============================================================================
# 9. SAFE SHUTDOWN
# ==============================================================================
print('Stopping supervisor, tunnel, and llama-server...')
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], check=False)
if 'supervisor_proc' in globals() and supervisor_proc.poll() is None:
    supervisor_proc.terminate()
if 'llama_proc' in globals() and llama_proc.poll() is None:
    llama_proc.terminate()
print('Processes stopped. Remember to click "Stop Session" in Kaggle to preserve GPU quota!')